# IC-1116 — Case study: is the backtest honest?

**ML & FinTech · 115-1 · 20261116 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Fill in both IDs; your client's two stocks are printed |
| 1 | 10 | The minimum-variance weight α **by hand** from your own variances |
| 2 | 12 | How sure is that α? Bootstrap it |
| 3 | 10 | One split, five folds, or time order? Compare with your partner |
| 4 | 5 | Hand in **one sheet of paper per pair** |

You may use any AI tool for the code. The graded part is the paper your pair hands in, and what
you say when called on. Your numbers come from your 學號; your partner's differ.

---

## The case

Your client holds two stocks, A and B. A manager has sent a backtest:

> *"The optimal allocation is **exactly 63.2% in A and 36.8% in B** — the variance-minimising split
> on last year's daily data. Our strategy model, validated on a random hold-out, predicts next-day
> direction with **61% accuracy**."*

Your client asks: **"How sure is that 63.2%? And can I trust 61%?"**

Your two stocks come from your 學號, so your α and your numbers are your own.

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/"
        "main/in-class-exercise/historical_stock_prices_2017.csv")
CAND = ["historical_stock_prices_2017.csv", "../historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
prices = pd.read_csv(path, index_col=0).dropna(axis=1); prices.index = pd.to_datetime(prices.index)
ret = np.log(prices).diff().dropna() * 100                # daily log returns, %

def draw_portfolio(student_id, k=10):
    key = lambda t: hashlib.sha256(f"{int(student_id)}-{t}".encode()).hexdigest()
    return sorted(sorted(prices.columns, key=key)[:k])

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
MY_PORTFOLIO = draw_portfolio(STUDENT_ID)
print("Client of", STUDENT_ID, "holds:", MY_PORTFOLIO)
A, B = MY_PORTFOLIO[0], MY_PORTFOLIO[1]
x, z = ret[A].values, ret[B].values
print(f"Stock A = {A}, stock B = {B}")
print(f"var(A) = {x.var(ddof=1):.3f}   var(B) = {z.var(ddof=1):.3f}   cov(A,B) = {np.cov(x, z)[0,1]:.3f}")

---
# Step 1 — α by hand (10 min)

The weight on A that minimises the variance of α·A + (1−α)·B is (ISLP §5.2)

α = ( σ²_B − σ_AB ) / ( σ²_A + σ²_B − 2 σ_AB ).

**Q1.** With the three numbers printed above, compute α by hand, to three decimals: α = ____ .
What is the variance of the portfolio at that α (var = α²σ²_A + (1−α)²σ²_B + 2α(1−α)σ_AB)? ____

Then run the cell: it computes the same thing.

In [ ]:
def alpha_hat(x, z):
    sx, sz, sxz = x.var(ddof=1), z.var(ddof=1), np.cov(x, z)[0,1]
    return (sz - sxz) / (sx + sz - 2*sxz)

a = alpha_hat(x, z)
print(f"alpha = {a:.3f}")

**Q2.** Does your hand value match? In one sentence: what would a client do with this number?

> _replace this line_

---
# Step 2 — How sure is α? The bootstrap (12 min)

Draw 1,000 bootstrap samples of the 250 days (with replacement, keeping A and B paired),
compute α on each, and report the **standard error** and the 2.5%–97.5% range.
Use `rng = np.random.default_rng(int(STUDENT_ID))`.

In [ ]:
# 2: bootstrap alpha; SE and 95% range



**Q3.** My α = ____ , bootstrap SE = ____ , 95% range ____ to ____ .

**Q4.** The manager quotes 63.2% to one decimal. Given your SE, how many digits of your α mean anything?
Would a client be wrong to hold 50/50 instead?

> _replace this line_

---
# Step 3 — Is 61% honest? (10 min)

Build a tiny strategy model for stock A: predict whether **tomorrow's** return is positive from
today's 5-day mean and 5-day standard deviation (code given). Then estimate its accuracy three ways:

1. one random 60/40 split (`train_test_split`, `random_state=1`), then again with `random_state=2` and `3`;
2. 5-fold cross-validation (`KFold(5, shuffle=True, random_state=0)`);
3. a **time-ordered** split: fit on the first 150 days, test on the rest.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score

s = ret[A]
F = pd.DataFrame({"m5": s.rolling(5).mean(), "v5": s.rolling(5).std()}).shift(0)
y = (s.shift(-1) > 0).astype(int)                 # tomorrow up?
ok = F.notna().all(axis=1) & s.shift(-1).notna()
F, y = F[ok], y[ok]
print(len(F), "days")

# 3: three random splits, 5-fold CV, and the time-ordered split



**Q5.** Accuracy: split 1 ____ , split 2 ____ , split 3 ____ ; 5-fold CV ____ ; time-ordered ____ .

**Q6.** Swap with your partner.

| | α | SE | time-ordered accuracy |
|---|---|---|---|
| me | | | |
| my partner | | | |

Why do the three random splits differ? Which of the five numbers would you report to the client as "the" accuracy, and is 61% from one random split worth anything?

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: α, its bootstrap SE, and the time-ordered accuracy.
3. **Two sentences, agreed between you**, answering the client's question and the trap.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1116/IC-1116.ipynb`
in your own repository before **12:10**.